# 01.5 — Per-(HVG flavor, holdout group) data preparation

This notebook generalizes `01.3_data_prep_all_holdouts_renorm.ipynb` along the HVG-flavor axis. It produces ONE combined dataset file per `(flavor, group)`, used by BOTH the scGen baseline and the IMPACT_CellOT model in BOTH IID and OOD modes — with the IID/OOD split handled at training time by `cellot/data/cell.py:split_cell_data_toggle_ood`.

## Design (see `research_log_2026-05-04.txt` for full justification)

- **HVG flavors (5):** `seurat`, `cell_ranger`, `seurat_v3`, `seurat_v3_paper`, `pearson_residuals`.
- **Holdout groups:** A=`cd8` (CL:0000625); B=`cd8_thymo` (CL:0000625 + CL:0000893); C=`tcell_subtypes` (CL:0000624 + CL:0000625 + CL:0000893); D=`cd4` (CL:0000624); **M2=`toggle_m2`** (non-classical + generic monocyte: CL:0000875, CL:0000576 — same as `09_data_prep_toggle_experiments.ipynb`).
- **Per-group HVG:** for each `(flavor, group)`, HVG is computed exactly once on the matched dataset with the per-group holdout removed, with `batch_key='species'`. The resulting top-1000 gene set is the fixed feature space for that group's IID and OOD configs and for both scGen and IMPACT.
- **One combined data file per (flavor, group):** `hvg_{flavor}_{gk}_v07.h5ad`. Both scGen and IMPACT consume it. IID/OOD splitting happens at training time via `datasplit.name=toggle_ood` with the same `random_state`, so the two models see byte-identical training cells in the same `(flavor, group, mode)` cell.
- **Cell pool: matched only (12,990 cells = 6,495 mouse + 6,495 human paired by `(cell_type, tissue)`).** This is **not** the large unmatched pool used in older monocyte toggle notebooks (`09`): every group (including M2) uses the **same** matched Tabula grid so IMPACT and scGen stay comparable. Monocytes appear only where that matched design includes them; cell counts for the M2 holdout are whatever the matcher yields, not the ~100k-scale toggle_m2 pool.

## Per-flavor HVG input dispatch (verified from scanpy 1.12 source)

| Flavor              | Input layer        | Why                                                |
|---------------------|--------------------|----------------------------------------------------|
| `seurat`            | `.X` (log-norm)    | Internally `expm1`s, computes dispersion in linear space, z-scores per mean bin. |
| `cell_ranger`       | `.X` (log-norm)    | Same `_highly_variable_genes_single_batch` path; bins by percentile of mean and MAD-normalizes dispersion. |
| `seurat_v3`         | `.layers['counts']`| `_highly_variable_genes_seurat_v3`, raises `UserWarning` if non-integers found. Uses regularized loess on log10(var) vs log10(mean). |
| `seurat_v3_paper`   | `.layers['counts']`| Identical to `seurat_v3` except multi-batch tie-breaker (median rank instead of average rank). |
| `pearson_residuals` | `.layers['counts']`| `scanpy.experimental.pp.highly_variable_genes`. Pearson residual = count-model fit; raw counts mandatory. |

## Outputs

- Up to 25 dataset files at `cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_{flavor}_{gk}_v07.h5ad` (5 flavors × 5 groups including **m2**), each ~50 MB when all are built. Use **`RUN_GROUP_KEYS` / `RUN_FLAVORS` in §1** to write only `m2` × (`seurat_v3`, `pearson_residuals`) without recomputing the full grid.
- Each file carries `.X` (log1p(normalize_total(counts))), `.obs[{condition, species, cell_type_ontology_term_id, cell_type, tissue_ontology_term_id, tissue, donor_id}]`, and `.var` indexed by ENSG ID for the per-(flavor, group) top-1000 gene subset. `.layers['counts']` is dropped before write (downstream training only consumes `.X`).
- Files are anndata-0.7 compatible after the round-trip in §7 — readable by `cellot/cellot_gpu/scripts/{train,evaluate}.py`.

## Kernel

Run with the `analysis` kernel (Python at `/n/home01/jzhou1125/miniforge3/envs/analysis/bin/python`, scanpy 1.12, has `scanpy.experimental.pp` and `seurat_v3_paper`). The CellOT env's scanpy 1.8.1 lacks both; the round-trip in §7 only uses anndata + h5py + numpy + pandas, all present in CellOT 0.7.

In [1]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")
from speciesot_helpers import (
    align_adatas_biomart_one2one,
    match_cells_by_celltype_tissue,
)

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
MOUSE_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/sampled_mouse_shared.h5ad"
HUMAN_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/sampled_human_shared.h5ad"

DATASET_DIR = os.path.join(
    BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg"
)
os.makedirs(DATASET_DIR, exist_ok=True)

CT_COL = "cell_type_ontology_term_id"
TISSUE_COL = "tissue_ontology_term_id"
N_HVG = 1000
RANDOM_STATE = 0

FLAVORS = ["seurat", "cell_ranger", "seurat_v3", "seurat_v3_paper", "pearson_residuals"]

GROUPS = {
    "a": {
        "name": "cd8",
        "description": "CD8 holdout only (mature CD8+ T cell)",
        "holdout_ids": ["CL:0000625"],
    },
    "b": {
        "name": "cd8_thymo",
        "description": "Combined holdout: CD8 + thymocyte",
        "holdout_ids": ["CL:0000625", "CL:0000893"],
    },
    "c": {
        "name": "tcell_subtypes",
        "description": "Combined holdout: CD4 + CD8 + thymocyte",
        "holdout_ids": ["CL:0000624", "CL:0000625", "CL:0000893"],
    },
    "d": {
        "name": "cd4",
        "description": "CD4 holdout only (mature CD4+ T cell)",
        "holdout_ids": ["CL:0000624"],
    },
    "m2": {
        "name": "toggle_m2",
        "description": "Non-classical + generic monocyte holdout (see 09_data_prep_toggle_experiments)",
        "holdout_ids": ["CL:0000875", "CL:0000576"],
    },
    "m1": {
        "name": "nonclassical_mono",
        "description": "Non-classical only",
        "holdout_ids": ["CL:0000875"],
    },
}

# Optional subset: e.g. RUN_GROUP_KEYS = {"m2"} and RUN_FLAVORS = ["seurat_v3", "pearson_residuals"]
# to build only monocyte M2 files without touching a–d or other flavors.
RUN_GROUP_KEYS = frozenset({"m1"})  # frozenset({"m2"}) or None for all groups
RUN_FLAVORS = ["pearson_residuals"]  # ["seurat_v3", "pearson_residuals"] or None for all flavors


def _active_groups(groups_dict):
    if RUN_GROUP_KEYS is None:
        return dict(groups_dict)
    wanted = set(RUN_GROUP_KEYS)
    missing = wanted - groups_dict.keys()
    if missing:
        raise KeyError(f"RUN_GROUP_KEYS contains unknown keys: {missing}")
    return {k: groups_dict[k] for k in sorted(wanted)}

def _active_flavors(all_flavors):
    if RUN_FLAVORS is None:
        return list(all_flavors)
    unknown = [f for f in RUN_FLAVORS if f not in all_flavors]
    if unknown:
        raise ValueError(f"RUN_FLAVORS contains unknown flavors: {unknown}")
    return list(RUN_FLAVORS)


GROUPS_ITER = _active_groups(GROUPS)
FLAVORS_ITER = _active_flavors(FLAVORS)
KEEP_OBS = [
    "condition",
    "species",
    "cell_type_ontology_term_id",
    "cell_type",
    "tissue_ontology_term_id",
    "tissue",
    "donor_id",
]

CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"

sc.settings.verbosity = 1
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("dataset dir:", DATASET_DIR)
print("flavors:", FLAVORS)
print("groups:", {k: v["holdout_ids"] for k, v in GROUPS.items()})
print("FLAVORS_ITER:", FLAVORS_ITER)
print("GROUPS_ITER keys:", list(GROUPS_ITER.keys()))

/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy: 1.12
anndata: 0.12.10
dataset dir: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg
flavors: ['seurat', 'cell_ranger', 'seurat_v3', 'seurat_v3_paper', 'pearson_residuals']
groups: {'a': ['CL:0000625'], 'b': ['CL:0000625', 'CL:0000893'], 'c': ['CL:0000624', 'CL:0000625', 'CL:0000893'], 'd': ['CL:0000624'], 'm2': ['CL:0000875', 'CL:0000576'], 'm1': ['CL:0000875']}
FLAVORS_ITER: ['pearson_residuals']
GROUPS_ITER keys: ['m1']


/tmp/ipykernel_2362221/2662195408.py:108: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)
/tmp/ipykernel_2362221/2662195408.py:109: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


## 1. Load raw counts (both species)

Promote `.raw` to `.X` so we have integer UMI/read counts for both species. Do NOT normalize yet — the raw counts are needed by the `seurat_v3`, `seurat_v3_paper`, and `pearson_residuals` flavors. Normalization happens in §4 after concatenation, leaving raw counts in `.layers['counts']` and log-normalized in `.X`.

Same source files and approach as `01.3` and `01.4`.

In [3]:
print("Loading source files and promoting .raw to .X ...")
mouse_full = sc.read_h5ad(MOUSE_H5AD)
human_full = sc.read_h5ad(HUMAN_H5AD)

assert mouse_full.raw is not None, "mouse_full.raw is missing; cannot proceed"
assert human_full.raw is not None, "human_full.raw is missing; cannot proceed"

mouse_all = mouse_full.raw.to_adata()
human_all = human_full.raw.to_adata()

mouse_all.obs = mouse_full.obs.copy()
human_all.obs = human_full.obs.copy()

mouse_all.X = mouse_all.X.astype("float32")
human_all.X = human_all.X.astype("float32")

for name, a in [("mouse_all", mouse_all), ("human_all", human_all)]:
    d = a.X.data if sp_sparse.issparse(a.X) else a.X.ravel()
    print(f"  {name}: {a.shape}   .X is raw counts? min={d.min():.0f} max={d.max():.0f} mean={d.mean():.2f}")

Loading source files and promoting .raw to .X ...
  mouse_all: (47807, 18024)   .X is raw counts? min=1 max=9523593 mean=168.76
  human_all: (58931, 61759)   .X is raw counts? min=1 max=5126079 mean=40.06


## 2. Ortholog alignment (raw counts)

Align mouse and human to a shared one-to-one ortholog axis using the BioMart-derived helper. Both AnnDatas come out with the SAME `var_names` (human ENSG IDs), still raw counts. ~14,451 shared orthologs are expected.

In [4]:
print("Aligning orthologs (BioMart one2one) ...")
mouse_aligned, human_aligned, ortholog_table = align_adatas_biomart_one2one(
    mouse_all, human_all
)

print(f"  mouse_aligned: {mouse_aligned.shape}")
print(f"  human_aligned: {human_aligned.shape}")
assert (mouse_aligned.var_names == human_aligned.var_names).all()

Aligning orthologs (BioMart one2one) ...
  mouse_aligned: (47807, 14451)
  human_aligned: (58931, 14451)


## 3. Match cells by `(cell_type, tissue)` — done once

`match_cells_by_celltype_tissue` only reads `.obs[cell_type_ontology_term_id]` and `.obs[tissue_ontology_term_id]`; it does NOT look at `.X` or `.layers`. So we can do it once on the full ortholog-aligned data and reuse the matched cell pool across all 5 flavors and all 4 groups.

The matched dataset has equal mouse/human counts per (cell_type, tissue) identity by construction — the property that makes the toggle_ood OOD evaluation cells balanced across species.

In [5]:
print("Matching cells by (cell_type, tissue) on the full ortholog-aligned data ...")
mouse_matched, human_matched = match_cells_by_celltype_tissue(
    mouse_aligned, human_aligned,
    cell_type_key=CT_COL,
    tissue_key=TISSUE_COL,
    seed=RANDOM_STATE,
)

print(f"  mouse_matched: {mouse_matched.shape}")
print(f"  human_matched: {human_matched.shape}")
assert mouse_matched.n_obs == human_matched.n_obs, (
    "match_cells_by_celltype_tissue should return paired counts per identity"
)

Matching cells by (cell_type, tissue) on the full ortholog-aligned data ...


  mouse_matched: (6495, 14451)
  human_matched: (6495, 14451)


## 4. Concat matched, snapshot raw counts to `.layers['counts']`, log-normalize `.X`

After this section `matched_full` carries:
- `.X` = `log1p(normalize_total(counts, target_sum=1e4))` — used by `seurat`/`cell_ranger` HVG and by all downstream training (scGen, IMPACT).
- `.layers['counts']` = the original raw integer counts — used by `seurat_v3`/`seurat_v3_paper`/`pearson_residuals` HVG.

Setting `.obs['species']` = `.obs['condition']` so `batch_key='species'` works in scanpy's HVG calls.

In [6]:
mouse_m = mouse_matched.copy()
human_m = human_matched.copy()
mouse_m.obs["condition"] = "mouse"
human_m.obs["condition"] = "human"

matched_full = ad.concat([mouse_m, human_m], join="inner")
matched_full.obs["species"] = matched_full.obs["condition"].values

raw_X = matched_full.X
if sp_sparse.issparse(raw_X):
    assert np.allclose(raw_X.data, np.round(raw_X.data)), "raw .X is not integer-valued before normalize"
    raw_int = raw_X.copy()
    raw_int.data = raw_int.data.astype(np.int32)
else:
    assert np.allclose(raw_X, np.round(raw_X)), "raw .X is not integer-valued before normalize"
    raw_int = raw_X.astype(np.int32)
matched_full.layers["counts"] = raw_int
print("layers[counts] stored as int32 sparse?", sp_sparse.issparse(matched_full.layers["counts"]),
      "dtype data" if sp_sparse.issparse(matched_full.layers["counts"]) else "dtype",
      matched_full.layers["counts"].data.dtype if sp_sparse.issparse(matched_full.layers["counts"]) else matched_full.layers["counts"].dtype)

sc.pp.normalize_total(matched_full, target_sum=1e4)
sc.pp.log1p(matched_full)

x_lognorm = matched_full.X.data if sp_sparse.issparse(matched_full.X) else matched_full.X.ravel()
x_counts = (
    matched_full.layers["counts"].data
    if sp_sparse.issparse(matched_full.layers["counts"])
    else matched_full.layers["counts"].ravel()
)
print(f"matched_full: {matched_full.shape}")
print(f"  .X (log-norm)         : min={x_lognorm.min():.4f}  max={x_lognorm.max():.4f}  mean={x_lognorm.mean():.4f}")
print(f"  .layers['counts'] (raw): min={x_counts.min():.0f}   max={x_counts.max():.0f}   mean={x_counts.mean():.2f}")
print("\nT-cell family present in matched dataset:")
T_CELL_IDS = {
    "CL:0000084": "T cell (broad)",
    "CL:0000893": "thymocyte",
    "CL:0000624": "CD4+ T cell",
    "CL:0000625": "CD8+ T cell",
}
for cid, name in T_CELL_IDS.items():
    mask = matched_full.obs[CT_COL].astype(str) == cid
    n_total = int(mask.sum())
    n_mouse = int((mask & (matched_full.obs["condition"] == "mouse")).sum())
    n_human = int((mask & (matched_full.obs["condition"] == "human")).sum())
    print(f"  {cid} ({name:15s}): {n_total:5d} total ({n_mouse:4d} mouse, {n_human:4d} human)")

layers[counts] stored as int32 sparse? True dtype data int32
matched_full: (12990, 14451)
  .X (log-norm)         : min=0.0003  max=8.8695  mean=1.2161
  .layers['counts'] (raw): min=1   max=4526500   mean=99.31

T-cell family present in matched dataset:
  CL:0000084 (T cell (broad) ):   204 total ( 102 mouse,  102 human)
  CL:0000893 (thymocyte      ):   910 total ( 455 mouse,  455 human)
  CL:0000624 (CD4+ T cell    ):   192 total (  96 mouse,   96 human)
  CL:0000625 (CD8+ T cell    ):   390 total ( 195 mouse,  195 human)


## 5. Per-flavor HVG dispatcher and `clean_adata` helper

`run_hvg_flavor` runs `sc.pp.highly_variable_genes` (or its experimental Pearson-residuals counterpart) with the right input layer for the given flavor. Returns a per-gene DataFrame with `highly_variable`, `rank`, and the per-flavor primary score column.

`clean_adata` strips `.uns`, `.obsm`, `.obsp`, `.varm`, `.varp`, and `.layers` to keep the written file small and anndata-0.7 friendly. Downstream `cellot/cellot_gpu/scripts/{train,evaluate}.py` only consume `.X` and `.obs`.

In [7]:
def run_hvg_flavor(adata, flavor, n_top=N_HVG, batch_key="species"):
    """Run HVG for the given flavor on a copy of adata. Returns a per-gene DataFrame
    indexed by `var_names` with columns: `score`, `highly_variable`, `rank`, `flavor`.

    Input layer per flavor:
      - 'seurat', 'cell_ranger'             : adata.X (must be log-normalized)
      - 'seurat_v3', 'seurat_v3_paper'      : adata.layers['counts'] (raw counts)
      - 'pearson_residuals'                 : adata.layers['counts'] (raw counts),
                                              via scanpy.experimental.pp
    """
    a = adata.copy()

    # Raw-count flavors require integer dtype in layers['counts']; .X is float32 from
    # promotion of .raw, but scanpy's safe-cast check treats float32->int as non-equivalent.
    # Re-cast in place after asserting the values really are integers.
    if flavor in ("seurat_v3", "seurat_v3_paper", "pearson_residuals") and "counts" in a.layers:
        L = a.layers["counts"]
        if sp_sparse.issparse(L):
            assert np.allclose(L.data, np.round(L.data)), "counts layer is not integer-valued"
            L_int = L.astype(np.int32)
        else:
            assert np.allclose(L, np.round(L)), "counts layer is not integer-valued"
            L_int = L.astype(np.int32)
        a.layers["counts"] = L_int

    if flavor in ("seurat", "cell_ranger"):
        sc.pp.highly_variable_genes(
            a, n_top_genes=n_top, flavor=flavor, batch_key=batch_key,
        )
        score_col = "dispersions_norm"

    elif flavor in ("seurat_v3", "seurat_v3_paper"):
        sc.pp.highly_variable_genes(
            a, n_top_genes=n_top, flavor=flavor, batch_key=batch_key,
            layer="counts",
        )
        score_col = "variances_norm"

    elif flavor == "pearson_residuals":
        from scanpy.experimental.pp import highly_variable_genes as hvg_pr
        hvg_pr(
            a, n_top_genes=n_top, flavor="pearson_residuals",
            batch_key=batch_key, layer="counts",
        )
        score_col = (
            "residual_variances"
            if "residual_variances" in a.var.columns
            else "highly_variable_rank"
        )

    else:
        raise ValueError(f"unknown flavor: {flavor!r}")

    df = a.var[["highly_variable"]].copy()
    df["score"] = a.var[score_col] if score_col in a.var.columns else np.nan
    df["flavor"] = flavor

    if "highly_variable_rank" in a.var.columns:
        # ranks can be non-integer (median across species batches in v3/pearson); keep as float
        df["rank"] = pd.to_numeric(a.var["highly_variable_rank"], errors="coerce").astype(float)
    else:
        df["rank"] = df["score"].rank(ascending=False, method="min").where(df["highly_variable"]).astype(float)
    return df


def clean_adata(adata):
    """Strip layers/obsm/obsp/uns/varm/varp; densify X; keep KEEP_OBS columns only.
    Downstream training/eval only need .X and .obs."""
    obs_cols = [c for c in KEEP_OBS if c in adata.obs.columns]
    X = adata.X
    if sp_sparse.issparse(X):
        X = np.array(X.todense())
    elif not isinstance(X, np.ndarray):
        X = np.array(X)
    return ad.AnnData(
        X=X.astype(np.float32),
        obs=adata.obs[obs_cols].copy(),
        var=pd.DataFrame(index=adata.var_names),
    )


print("Helpers defined.")

Helpers defined.


## 6. Per-(flavor, group) HVG selection and write `hvg_{flavor}_{gk}` files

Runs over **`GROUPS_ITER` × `FLAVORS_ITER` from §1** (defaults: all groups including **m2**, all five flavors).

For each `(flavor, group)`:

1. Define `train_mask = ~matched_full.obs[CT_COL].isin(group['holdout_ids'])`. The training-eligible cells are those NOT in the per-group holdout.
2. Compute HVG using `run_hvg_flavor` on `matched_full[train_mask]`. Pull the top-`N_HVG` (1000) gene names.
3. Subset `matched_full[:, hvg_genes]` — KEEPS the holdout cells in the file (they are needed at training time for `toggle_ood` IID/OOD splitting and for evaluation).
4. `clean_adata(...)` and write to `hvg_{flavor}_{gk}.h5ad`.
5. Save the per-(flavor, group) HVG table to `hvg_flavor_outputs/hvg_{GROUPKEY}_{flavor}_perGroup.csv` for diagnostics. Distinct from `01.4`'s Group-A-only outputs by the `_perGroup` suffix.

Note: `01.4` already wrote per-flavor HVG tables for Group A (without the `_perGroup` suffix and excluding `cell_ranger`). Those remain untouched; `01.5` writes new tables with the `_perGroup` suffix.

In [8]:
HVG_TABLE_DIR = os.path.join(BASE_DIR, "speciesOT/baseline/analysis/hvg_flavor_outputs")
os.makedirs(HVG_TABLE_DIR, exist_ok=True)

summary_rows = []

for gk, g in GROUPS_ITER.items():
    holdout_ids = set(g["holdout_ids"])
    train_mask = ~matched_full.obs[CT_COL].astype(str).isin(holdout_ids)
    holdout_mask = matched_full.obs[CT_COL].astype(str).isin(holdout_ids)
    n_train = int(train_mask.sum())
    n_holdout = int(holdout_mask.sum())

    print(f"\n{'=' * 70}")
    print(f"GROUP {gk.upper()} ({g['name']}): {g['description']}")
    print(f"  holdout_ids: {sorted(holdout_ids)}")
    print(f"  matched cells: {matched_full.n_obs}  ->  train-eligible {n_train}, holdout {n_holdout}")

    train_subset = matched_full[train_mask].copy()

    for flavor in FLAVORS_ITER:
        print(f"  flavor={flavor!r} ...", end=" ", flush=True)
        try:
            hvg_df = run_hvg_flavor(train_subset, flavor)
        except Exception as e:
            print(f"FAILED: {e}")
            continue

        hv_genes = (
            hvg_df[hvg_df["highly_variable"]]
            .sort_values("rank", na_position="last")
            .index.tolist()
        )
        if len(hv_genes) > N_HVG:
            hv_genes = hv_genes[:N_HVG]

        hvg_table_path = os.path.join(
            HVG_TABLE_DIR, f"hvg_{gk}_{flavor}_perGroup.csv"
        )
        hvg_df.to_csv(hvg_table_path)

        sub = matched_full[:, hv_genes].copy()
        cleaned = clean_adata(sub)
        out_path = os.path.join(DATASET_DIR, f"hvg_{flavor}_{gk}.h5ad")
        cleaned.write_h5ad(out_path)
        print(
            f"{len(hv_genes)} HVG -> {os.path.basename(out_path)} "
            f"({cleaned.n_obs} cells x {cleaned.n_vars} genes)"
        )

        summary_rows.append({
            "group": gk,
            "group_name": g["name"],
            "flavor": flavor,
            "n_hvg": len(hv_genes),
            "n_train_cells": n_train,
            "n_holdout_cells": n_holdout,
            "n_total_cells": cleaned.n_obs,
            "out_path": out_path,
        })

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(HVG_TABLE_DIR, "hvg_flavor_perGroup_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\nSummary table: {summary_csv}")
summary_df


GROUP A (cd8): CD8 holdout only (mature CD8+ T cell)
  holdout_ids: ['CL:0000625']
  matched cells: 12990  ->  train-eligible 12600, holdout 390
  flavor='seurat' ... 1000 HVG -> hvg_seurat_a.h5ad (12990 cells x 1000 genes)
  flavor='cell_ranger' ... 1000 HVG -> hvg_cell_ranger_a.h5ad (12990 cells x 1000 genes)
  flavor='seurat_v3' ... 1000 HVG -> hvg_seurat_v3_a.h5ad (12990 cells x 1000 genes)
  flavor='seurat_v3_paper' ... 1000 HVG -> hvg_seurat_v3_paper_a.h5ad (12990 cells x 1000 genes)
  flavor='pearson_residuals' ... 1000 HVG -> hvg_pearson_residuals_a.h5ad (12990 cells x 1000 genes)

GROUP B (cd8_thymo): Combined holdout: CD8 + thymocyte
  holdout_ids: ['CL:0000625', 'CL:0000893']
  matched cells: 12990  ->  train-eligible 11690, holdout 1300
  flavor='seurat' ... 1000 HVG -> hvg_seurat_b.h5ad (12990 cells x 1000 genes)
  flavor='cell_ranger' ... 1000 HVG -> hvg_cell_ranger_b.h5ad (12990 cells x 1000 genes)
  flavor='seurat_v3' ... 1000 HVG -> hvg_seurat_v3_b.h5ad (12990 cells x

,group,group_name,flavor,n_hvg,n_train_cells,n_holdout_cells,n_total_cells,out_path
0,a,cd8,seurat,1000,12600,390,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
1,a,cd8,cell_ranger,1000,12600,390,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
2,a,cd8,seurat_v3,1000,12600,390,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
3,a,cd8,seurat_v3_paper,1000,12600,390,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
4,a,cd8,pearson_residuals,1000,12600,390,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
5,b,cd8_thymo,seurat,1000,11690,1300,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
6,b,cd8_thymo,cell_ranger,1000,11690,1300,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
7,b,cd8_thymo,seurat_v3,1000,11690,1300,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
8,b,cd8_thymo,seurat_v3_paper,1000,11690,1300,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...
9,b,cd8_thymo,pearson_residuals,1000,11690,1300,12990,/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT...


## 7. Round-trip via the CellOT env for `anndata 0.7` compatibility

The 20 files just written by anndata 1.x carry empty placeholder groups (`/layers`, `/obsm`, `/obsp`, `/uns`, `/varm`, `/varp`) and top-level `encoding-type`/`encoding-version` attributes that anndata 0.7 (in the `CellOT` conda env, used by `train.py` and `evaluate.py`) cannot read. Categorical `.obs` columns are also serialized in 0.2.0 format (nested groups with `categories` + `codes`) instead of the flat 0.1.0 layout that anndata 0.7 expects.

Fix (same pattern as `01.3` cells 18–19):
1. Strip empty groups via version-agnostic `h5py`.
2. Shell out to the **CellOT env's own Python interpreter** to reconstruct each AnnData from `h5py` and `write()` it via anndata 0.7, which emits the 0.1.0 obs layout.

Outputs are renamed to `hvg_{flavor}_{gk}_v07.h5ad` after the round-trip.

In [9]:
hvg_files = sorted(
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.startswith("hvg_") and f.endswith(".h5ad") and "_v07" not in f
)

v07_paths = []
for src in hvg_files:
    dst = src.replace(".h5ad", "_v07.h5ad")
    if os.path.exists(dst):
        os.remove(dst)
    shutil.copy2(src, dst)
    v07_paths.append(dst)

print(f"Copied {len(v07_paths)} files to *_v07.h5ad. Now stripping/rewriting via CellOT env ...")

strip_script = r"""
import sys, h5py

EMPTY_GROUPS = ["layers", "obsm", "obsp", "uns", "varm", "varp"]
for path in sys.argv[1:]:
    with h5py.File(path, "r+") as f:
        for g in EMPTY_GROUPS:
            if g in f and len(f[g].keys()) == 0:
                del f[g]
        for attr in ("encoding-type", "encoding-version"):
            if attr in f.attrs:
                del f.attrs[attr]
    print("  stripped:", path)
"""

subprocess.run([CELLOT_PY, "-c", strip_script, *v07_paths], check=True)

rewrite_script = r"""
import sys, os
import h5py
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

def _decode(x):
    return x.decode() if isinstance(x, (bytes, np.bytes_)) else x

def load_obs(f):
    obs_grp = f["obs"]
    idx_key = _decode(obs_grp.attrs["_index"]) if "_index" in obs_grp.attrs else "index"
    index = [_decode(x) for x in obs_grp[idx_key][:]]
    cols = {}
    for name in obs_grp.keys():
        if name == idx_key:
            continue
        node = obs_grp[name]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats = [_decode(c) for c in node["categories"][:]]
            cols[name] = pd.Categorical.from_codes(node["codes"][:], categories=cats)
        else:
            arr = node[:]
            if arr.dtype.kind in ("O", "S"):
                arr = np.array([_decode(x) for x in arr])
            cols[name] = arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx_key))

def load_var(f):
    var_grp = f["var"]
    idx_key = _decode(var_grp.attrs["_index"]) if "_index" in var_grp.attrs else "index"
    index = [_decode(x) for x in var_grp[idx_key][:]]
    cols = {}
    for name in var_grp.keys():
        if name == idx_key:
            continue
        node = var_grp[name]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats = [_decode(c) for c in node["categories"][:]]
            cols[name] = pd.Categorical.from_codes(node["codes"][:], categories=cats)
        else:
            arr = node[:]
            if arr.dtype.kind in ("O", "S"):
                arr = np.array([_decode(x) for x in arr])
            cols[name] = arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx_key))

def load_X(f):
    node = f["X"]
    if isinstance(node, h5py.Group):
        data = node["data"][:]
        indices = node["indices"][:]
        indptr = node["indptr"][:]
        shape = tuple(node.attrs.get("shape", node.attrs.get("h5sparse_shape")))
        encoding = _decode(node.attrs.get("encoding-type", b"csr_matrix"))
        if "csc" in encoding:
            return sparse.csc_matrix((data, indices, indptr), shape=shape)
        return sparse.csr_matrix((data, indices, indptr), shape=shape)
    return node[:]

for path in sys.argv[1:]:
    with h5py.File(path, "r") as f:
        obs_df = load_obs(f)
        var_df = load_var(f)
        X = load_X(f)
    adata = ad.AnnData(X=X, obs=obs_df, var=var_df)
    os.remove(path)
    adata.write(path)
    print("  rewrote:", path, "| anndata", ad.__version__, "| shape", adata.shape)
"""

subprocess.run([CELLOT_PY, "-c", rewrite_script, *v07_paths], check=True)

print("\nAll _v07 files are now CellOT-env compatible.")

Copied 27 files to *_v07.h5ad. Now stripping/rewriting via CellOT env ...
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_cell_ranger_a_v07.h5ad
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_cell_ranger_b_v07.h5ad
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_cell_ranger_c_v07.h5ad
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_cell_ranger_d_v07.h5ad
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_cell_ranger_m2_v07.h5ad
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_a_v07.h5ad
  stripped: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets

## 8. Verification

For each `(flavor, group)` `_v07.h5ad`:
- Confirm anndata 0.7 (CellOT env) can read it.
- Spot-check `.X` is log-normalized in a sane range.
- Spot-check `.obs` has `condition`, `species`, `cell_type_ontology_term_id` columns.
- Confirm holdout cells are present so `toggle_ood` can split them at training time.
- Confirm `condition` has both `mouse` and `human` so `compute_scgen_shift` works.

Also lists the 20 `_v07.h5ad` files with sizes for a quick sanity sweep.

In [10]:
v07_files = sorted(
    f for f in os.listdir(DATASET_DIR)
    if f.startswith("hvg_") and f.endswith("_v07.h5ad")
)
print(f"Found {len(v07_files)} _v07 files in {DATASET_DIR}:")
for f in v07_files:
    sz = os.path.getsize(os.path.join(DATASET_DIR, f)) / (1024 * 1024)
    print(f"  {f:60s}  {sz:7.1f} MB")

verify_script = r"""
import sys, json
import numpy as np
import anndata as ad

problems = []
for path in sys.argv[1:]:
    try:
        a = ad.read(path)
    except Exception as e:
        problems.append({"path": path, "error": f"read failed: {e}"})
        continue

    info = {"path": path, "shape": a.shape}
    info["X_min"] = float(np.nanmin(a.X))
    info["X_max"] = float(np.nanmax(a.X))
    info["X_mean"] = float(np.nanmean(a.X))
    info["obs_cols"] = sorted(a.obs.columns.tolist())
    if "condition" in a.obs.columns:
        info["condition_counts"] = a.obs["condition"].astype(str).value_counts().to_dict()
    if "cell_type_ontology_term_id" in a.obs.columns:
        ct_counts = a.obs["cell_type_ontology_term_id"].astype(str).value_counts()
        info["n_distinct_celltypes"] = int(ct_counts.size)
    print(json.dumps(info))

if problems:
    print("PROBLEMS:")
    for p in problems:
        print(json.dumps(p))
    sys.exit(1)
"""

subprocess.run(
    [CELLOT_PY, "-c", verify_script, *(os.path.join(DATASET_DIR, f) for f in v07_files)],
    check=True,
)
print("\nAll _v07 files verified readable in the CellOT (anndata 0.7) env.")

Found 27 _v07 files in /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg:
  hvg_cell_ranger_a_v07.h5ad                                       50.6 MB
  hvg_cell_ranger_b_v07.h5ad                                       50.6 MB
  hvg_cell_ranger_c_v07.h5ad                                       50.6 MB
  hvg_cell_ranger_d_v07.h5ad                                       50.6 MB
  hvg_cell_ranger_m2_v07.h5ad                                      50.6 MB
  hvg_pearson_residuals_a_v07.h5ad                                 50.6 MB
  hvg_pearson_residuals_atlas_full_v07.h5ad                        33.6 MB
  hvg_pearson_residuals_b_v07.h5ad                                 50.6 MB
  hvg_pearson_residuals_c_v07.h5ad                                 50.6 MB
  hvg_pearson_residuals_d_v07.h5ad                                 50.6 MB
  hvg_pearson_residuals_m2_v07.h5ad                                50.6 MB
  hvg_seurat_a_v07.h5ad                            

## 9. Next steps

1. Run `speciesOT/scripts/generate_hvg_flavor_configs.py`. Default output is still **4 T-cell groups × 5 flavors** (80 model cells): configs under `cellot/cellot_gpu/results/hvg_{flavor}_{gk}_{mode}/` plus `sbatch/{train,eval}/`. For **monocyte M2** with **`seurat_v3` + `pearson_residuals`** only, use **`--m2-two-flavors`** (adds **`sbatch/eval_dataspace/`** for gene-space `evals_ood_data_space`, same convention as CD8).
2. Submit `train_scgen` sbatches first (CPU partition, ~30–60 min each).
3. Submit `train_impact` sbatches with `--dependency=afterok:<scgen_jobid>` per `(flavor, group, mode)` cell.
4. Submit eval sbatches after training. For slides / Pearson bars in gene space, submit the **`eval_dataspace`** scripts (when generated).